# ask

> answer from the vault, with citations back into it

In [ ]:
#| default_exp ask

In [ ]:
#| hide
from nbdev.showdoc import *

Retrieval hands the model numbered sections; the model cites `[n]`; `_cited` resolves those back to
`node_id`s you can `read()`. That round trip is what separates an answer you can check from one you
can only believe.

In [ ]:
#| export
import os, re
from fastcore.all import AttrDict, L, patch
from vishalakshi.core import Vault, tidy_bc

In [ ]:
#| export
VAULT_SP = """You answer questions from a personal research vault.

You are given numbered sections retrieved from the user's own corpus — papers, web pages,
transcripts, files and their own notes. Answer only from those sections.

Rules:
- Cite every claim with the bracketed number of the section it came from, like [2]. A sentence
  drawing on two sections cites both.
- If the sections do not answer the question, say exactly what is missing rather than filling the
  gap from memory. A vault that admits a hole is useful; one that guesses is not.
- Sections marked RELATED were reached by association, not by matching the question. Use them for
  context or to point somewhere worth reading next, and say so when you do.
- Prefer the user's own notes when they conflict with a source, and flag the disagreement."""

dflt_model = os.getenv('VISHALAKSHI_MODEL', 'gemma-3-4b-it-int4')

def fmt_context(ctx,                 # AttrDict from Vault.context()
                max_chars:int=4000,  # chars kept per section
                related:bool=True,   # include the associative leg
) -> str:
    """Render retrieved sections as a numbered, citable block.

    The numbering is the contract with the model: `[n]` in the answer maps to `ctx.results[n-1]`,
    which is what makes an answer checkable against the vault instead of merely plausible."""
    parts = []
    for i, r in enumerate(ctx.results, 1):
        src = r.filename or r.doc_id
        pg = f", pages {r.pages[0]}–{r.pages[1]}" if r.pages and r.pages[0] is not None else ''
        parts.append(f"[{i}] {tidy_bc(r.breadcrumb)}\n(source: {src}{pg})\n\n{(r.text or '')[:max_chars]}")
    if related and ctx.related:
        rel = '\n'.join(f"- {tidy_bc(r.breadcrumb)} (reached by {r.via})" for r in ctx.related)
        parts.append(f"RELATED — not retrieved by the question, but connected to what was:\n{rel}")
    return '\n\n---\n\n'.join(parts)

def mk_prompt(question:str, ctx, max_chars:int=4000, related:bool=True) -> str:
    'The user turn: the numbered sections, then the question.'
    body = fmt_context(ctx, max_chars=max_chars, related=related)
    if not body.strip(): return f"The vault returned nothing for this question.\n\nQuestion: {question}"
    return f"{body}\n\n---\n\nQuestion: {question}"

In [ ]:
#| export
_CITE = re.compile(r'\[(\d+)\]')

def _cited(answer:str, results) -> list:
    'The sections an answer actually cited, in citation order — the audit trail for a claim.'
    out, seen = [], set()
    for m in _CITE.finditer(answer or ''):
        i = int(m.group(1))
        if 1 <= i <= len(results) and i not in seen:
            seen.add(i)
            r = results[i-1]
            out.append(dict(n=i, node_id=r.node_id, title=r.title, breadcrumb=tidy_bc(r.breadcrumb),
                            source=r.filename, doc_id=r.doc_id))
    return out

In [ ]:
#| export
@patch
def chat(self:Vault, model:str=None, sp:str=VAULT_SP, **kw):
    """A `rishi.Chat` bound to this vault's system prompt, cached on the vault.

    The model id picks the backend by itself: a `litert-community` id or `.litertlm` runs on
    LiteRT, a `.gguf` on llama.cpp, an `mlx-community` id on MLX, and a hosted name like
    `claude-sonnet-5` through fastllm. Local and cloud are the same call."""
    key = (model or dflt_model, sp)
    if getattr(self, '_chat_key', None) != key:
        from rishi.core import Chat
        self._chat, self._chat_key = Chat(model or dflt_model, sp=sp, **kw), key
    return self._chat

@patch
def ask(self:Vault,
        question:str,          # what you want to know
        model:str=None,        # rishi model id; None -> $VISHALAKSHI_MODEL
        sections:int=6,        # operative sections retrieved
        related:int=6,         # associative sections offered as leads
        kind=None,             # restrict retrieval to one or more KINDS
        max_chars:int=4000,    # chars of each section shown to the model
        sp:str=VAULT_SP,       # system prompt
        fresh:bool=True,       # start a new conversation rather than continuing the last
        **kw                   # forwarded to Vault.context
) -> AttrDict:
    """Retrieve, then answer with citations back into the vault.

    Returns `AttrDict(question, answer, cited, context, prompt, encoder)`. `cited` resolves the
    `[n]` markers in the answer back to `node_id`s you can `read()`, so every claim is one call
    away from the text it came from. `context` is the full retrieval, kept so you can inspect what
    the model was and was not shown."""
    ctx = self.context(question, sections=sections, related=related, kind=kind, **kw)
    prompt = mk_prompt(question, ctx, max_chars=max_chars, related=bool(related))
    ch = self.chat(model=model, sp=sp)
    if fresh: ch.hist = []
    from rishi.core import resp_text
    answer = resp_text(ch(prompt))
    return AttrDict(question=question, answer=answer, cited=_cited(answer, ctx.results),
                    context=ctx, prompt=prompt, encoder=self.enc.note,
                    model=(model or dflt_model), usage=getattr(ch, 'use', None))

@patch
def explain(self:Vault, node_id:str, model:str=None, max_chars:int=6000, **kw) -> AttrDict:
    'Have a model explain one section in the context of what the vault connects it to.'
    sec = self.read(node_id, max_chars=max_chars)
    rel = self.related(node_id, limit=6)
    leads = '\n'.join(f"- {tidy_bc(r['breadcrumb'])}" for r in rel)
    prompt = (f"Section: {sec.get('title','')}\n\n{sec.get('text','')}\n\n"
              f"Other sections in the vault that read like it:\n{leads}\n\n"
              "Explain this section, then say what the related sections add or contradict.")
    ch = self.chat(model=model, sp=VAULT_SP, **kw)
    ch.hist = []
    from rishi.core import resp_text
    return AttrDict(node_id=node_id, answer=resp_text(ch(prompt)), section=sec, related=rel)